E-9: Scrivere extract(model, loader, which) con which ∈ {"z_cls", "z_mean", "h_pen"}, che restituisce un array (n, d) e gli indici
nell'ordine originale. Con model.eval(), dentro torch.inference_mode(). Verificare il determinismo.

In [ ]:
# E-9 — Head del modello

class Head(nn.Module):

    def __init__(self, d=64, K=20):
        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(d, d),
            nn.GELU(),
            nn.Linear(d, d)
        )

        self.classifier = nn.Linear(d, K)

    def forward(self, z):
        h = self.mlp(z)
        return self.classifier(h), h


# Creiamo la head
head = Head(d=64, K=20)

print(head)

Head(
  (mlp): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=64, out_features=64, bias=True)
  )
  (classifier): Linear(in_features=64, out_features=20, bias=True)
)


In [ ]:
# E-9 — Dataset delle sequenze

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader


class FlowSequenceDataset(Dataset):

    def __init__(self, frame):

        self.direction = np.stack(
            frame["splt_direction"].to_numpy()
        ).astype(np.float32)

        self.ps = np.stack(
            frame["splt_ps"].to_numpy()
        ).astype(np.float32)

        self.piat = np.stack(
            frame["splt_piat_ms"].to_numpy()
        ).astype(np.float32)

        self.mask = np.stack(
            frame["splt_mask"].to_numpy()
        ).astype(np.bool_)

        # Indici originali del DataFrame
        self.indices = frame.index.to_numpy()

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):

        # (20, 3)
        x = np.stack(
            [
                self.direction[i],
                self.ps[i],
                self.piat[i]
            ],
            axis=1
        )

        # PyTorch: True = posizione mascherata
        pad_mask = ~self.mask[i]

        return (
            torch.from_numpy(x),
            torch.from_numpy(pad_mask),
            int(self.indices[i])
        )


print("Classe Dataset creata correttamente.")

Classe Dataset creata correttamente.


In [ ]:
# E-9 — Creazione dei Dataset

train_df = df[df["split"] == "train"]
val_df   = df[df["split"] == "val"]
test_df  = df[df["split"] == "test"]
probe_df = df[df["split"] == "probe"]

train_dataset = FlowSequenceDataset(train_df)
val_dataset   = FlowSequenceDataset(val_df)
test_dataset  = FlowSequenceDataset(test_df)
probe_dataset = FlowSequenceDataset(probe_df)

print("Train:", len(train_dataset))
print("Val:  ", len(val_dataset))
print("Test: ", len(test_dataset))
print("Probe:", len(probe_dataset))

Train: 393783
Val:   84383
Test:  56255
Probe: 28128


In [ ]:
# E-9 — DataLoader

batch_size = 256

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

probe_loader = DataLoader(
    probe_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("DataLoader creati correttamente.")

DataLoader creati correttamente.


In [ ]:
# E-9 — Estrazione delle rappresentazioni

def extract(model, head, loader, which):

    model.eval()
    head.eval()

    representations = []
    indices = []

    with torch.inference_mode():

        for x, pad_mask, batch_indices in loader:

            # Encoder
            out = model(x, pad_mask)

            # Rappresentazione CLS
            if which == "z_cls":

                z = out[:, 0, :]

            # Media delle sole posizioni valide
            elif which == "z_mean":

                valid = ~pad_mask

                z = (
                    out[:, 1:, :] * valid.unsqueeze(-1)
                ).sum(dim=1) / valid.sum(dim=1, keepdim=True)

            # Rappresentazione prima del classificatore
            elif which == "h_pen":

                z_cls = out[:, 0, :]
                _, z = head(z_cls)

            else:
                raise ValueError(
                    "which deve essere 'z_cls', 'z_mean' oppure 'h_pen'"
                )

            representations.append(z.cpu().numpy())
            indices.append(batch_indices.numpy())

    representations = np.concatenate(representations, axis=0)
    indices = np.concatenate(indices, axis=0)

    return representations, indices


print("Funzione extract creata correttamente.")

Funzione extract creata correttamente.


In [ ]:
# E-9 — Verifica determinismo dell'estrazione

z_cls_1, idx_1 = extract(
    model, head, extract_train_loader, "z_cls"
)

z_cls_2, idx_2 = extract(
    model, head, extract_train_loader, "z_cls"
)

print("z_cls identico:", np.array_equal(z_cls_1, z_cls_2))
print("indici identici:", np.array_equal(idx_1, idx_2))

if np.array_equal(z_cls_1, z_cls_2) and np.array_equal(idx_1, idx_2):
    print("✓ Test di determinismo SUPERATO")
else:
    print("✗ Test di determinismo FALLITO")

z_cls identico: True
indici identici: True
✓ Test di determinismo SUPERATO


In [ ]:
# E-9 — DataLoader per estrazione deterministica

extract_train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("Extract DataLoader creato con shuffle=False.")

Extract DataLoader creato con shuffle=False.
